# gramlab — writer swap (T4, ≈1 h): a pretrained 0.5B model fine-tuned to read passages
Input: **gramlab** (Version 3: `data/rag_qa.jsonl`, BM25 index). GPU T4 x2, Internet on. *Save & Run All*.
Qwen2.5-0.5B-Instruct + LoRA on the 27k retrieval-conditioned dialogues (2 epochs, ~45 min), held-out evaluation, the usual
sample questions through BM25 → model, merged model saved (~1 GB) for the chat notebook.

In [ ]:
import os, subprocess, sys, time, glob, shutil, json
T0 = time.time()
def sh(cmd, must=True):
    print('\n$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True)
    if must and r.returncode != 0:
        raise SystemExit(f'command failed ({r.returncode}): {cmd}')
os.chdir('/kaggle/working')
if not os.path.exists('gramlab'):
    sh('git clone --depth 1 https://github.com/furevych/gramlab.git')
os.chdir('/kaggle/working/gramlab')
os.makedirs('work', exist_ok=True); os.makedirs('results', exist_ok=True); os.makedirs('data', exist_ok=True)

root = sorted(glob.glob('/kaggle/input/**/data/rag_qa.jsonl', recursive=True))[0].split('/data/')[0]
shutil.copy(f'{root}/data/rag_qa.jsonl', 'data/')
if not os.path.exists('work/bm25'):
    shutil.copytree(f'{root}/work/bm25', 'work/bm25')
sh('pip uninstall -y -q torchao', must=False)        # Kaggle image ships torchao 0.10, which transformers refuses to import next to
try:
    import peft
except ImportError:
    sh('pip install -q peft')
import torch; print('torch', torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

In [ ]:
sh('python scripts/finetune_hf.py --base Qwen/Qwen2.5-0.5B-Instruct --corpus data/rag_qa.jsonl --out work/runs/writer_hf --epochs 2 --max_hours 1.5')
print(json.load(open('work/runs/writer_hf/gramlab.json'))['eval'])
print(f'elapsed {(time.time()-T0)/60:.0f} min')

## Sample questions: BM25 → 0.5B writer

In [ ]:
import torch
sys.path.insert(0, '/kaggle/working/gramlab')
from transformers import AutoTokenizer, AutoModelForCausalLM
from gramlab.retrieval import BM25Index, format_rag_user
index = BM25Index.load('work/bm25')
tok = AutoTokenizer.from_pretrained('work/runs/writer_hf'); hf = AutoModelForCausalLM.from_pretrained('work/runs/writer_hf', torch_dtype=torch.float16).cuda().eval()
SYSTEM = json.load(open('work/runs/writer_hf/gramlab.json'))['system']
def ask(q, k=1):
    p = index.retrieve(q, k=k, max_words=90) or [{'title': '(no match)', 'text': 'No passage found.'}]
    chat = [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': format_rag_user(p, q)}]
    ids = tok.apply_chat_template(chat, add_generation_prompt=True, return_tensors='pt').cuda()
    with torch.no_grad():
        out = hf.generate(ids, max_new_tokens=80, do_sample=False, repetition_penalty=1.1, pad_token_id=tok.pad_token_id)
    return f"[read: {p[0]['title']}] " + tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()
for q in ['Tell me about Belgrade.', 'Who was Napoleon?', 'What is the capital of Japan?', 'Who was Albert Einstein?', 'What is photosynthesis?',
          'who is churchhill', 'Which river flows through Vienna?', 'When was Albert Einstein born?', 'Tell me about Zxqv Blorptown.']:
    print('Q:', q); print('A:', ask(q)); print()

In [ ]:
shutil.rmtree('/kaggle/working/gramlab/.git', ignore_errors=True)
shutil.rmtree('/kaggle/working/gramlab/work/bm25', ignore_errors=True)      # lives in the gramlab output already
os.system('du -sh /kaggle/working/gramlab/work/runs/* 2>/dev/null')
print(f'total {(time.time()-T0)/60:.0f} min')